# Walmart - Pharmacy Consultation Privacy for Patient Comfort

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_consultation = pd.read_csv('../Data/013/fct_consultations_2.csv', parse_dates=['consultation_date'])

pl_consultation = pl.read_csv('../Data/013/fct_consultations_2.csv', try_parse_dates=True)

# Preguntas 1

### ¿Cuáles son los nombres de las 3 farmacias que realizaron el menor número de consultas en julio de 2024? Esto nos ayudará a identificar las ubicaciones con espacios de consulta potencialmente menos concurridos.

```SQL
SELECT
    pharmacy_name,
    COUNT(consultation_id) AS num_consultas
FROM fct_consultations
WHERE ((EXTRACT(MONTH FROM consultation_date) = 7) AND
       (EXTRACT(YEAR FROM consultation_date) = 2024))
GROUP BY pharmacy_name
ORDER BY num_consultas ASC
LIMIT 3;
```

In [6]:
julio = df_consultation[
    (df_consultation['consultation_date'].dt.month == 7) &
    (df_consultation['consultation_date'].dt.year == 2024)
].groupby('pharmacy_name').agg(
    num_consultas = ('consultation_id', 'count')
).reset_index()

res = julio.sort_values('num_consultas', ascending=True).head(3)

In [8]:
res = pl_consultation.filter(
    (pl.col('consultation_date').dt.month() == 7) &
    (pl.col('consultation_date').dt.year() == 2024)
).group_by('pharmacy_name').agg(
    pl.len().alias('num_consultas')
).sort('num_consultas', descending=False).head(3)

# Pregunta 2

### Para las farmacias identificadas en la pregunta anterior (es decir, las 3 farmacias con el menor número de consultas en julio de 2024), ¿cuál es la versión en mayúsculas de los tipos de sala de consulta disponibles? Comprender los tipos de sala puede proporcionar información sobre las características de privacidad ofrecidas.

```SQL
WITH top_3_farmacias AS(
    SELECT
        pharmacy_name
    FROM fct_consultations
    WHERE ((EXTRACT(MONTH FROM consultation_date) = 7) AND
           (EXTRACT(YEAR FROM consultation_date) = 2024))
    GROUP BY pharmacy_name
    ORDER BY COUNT(consultation_id) ASC
    LIMIT 3
)
SELECT
    DISTINCT UPPER(consultation_room_type) AS sala_mayusculas
FROM fct_consultations
WHERE pharmacy_name IN (SELECT pharmacy_name FROM top_3_farmacias);
```

In [28]:
julio = df_consultation[
    (df_consultation['consultation_date'].dt.month == 7) &
    (df_consultation['consultation_date'].dt.year == 2024)
].groupby('pharmacy_name').agg(
    num_consultas = ('consultation_id', 'count')
).reset_index()

res = julio.sort_values('num_consultas', ascending=True).head(3)

top_farmacias = res['pharmacy_name'].tolist()

salas_finales = (
    df_consultation[df_consultation['pharmacy_name'].isin(top_farmacias)]
    ['consultation_room_type']
    .str.upper()
    .unique()
)

In [31]:
res = pl_consultation.filter(
    (pl.col('consultation_date').dt.month() == 7) &
    (pl.col('consultation_date').dt.year() == 2024)
).group_by('pharmacy_name').agg(
    pl.len().alias('num_consultas')
).sort('num_consultas', descending=False).head(3)

top_farmacias = res['pharmacy_name'].to_list()

salas_finales = (
    pl_consultation.filter(
        pl.col('pharmacy_name').is_in(top_farmacias)
    ).select(
        pl.col('consultation_room_type').str.to_uppercase().unique()
    )
)

# Pregunta 3

### Hasta ahora, hemos identificado las 3 farmacias con menos consultas en julio de 2024. Entre estas 3 farmacias, ¿cuál es el puntaje de nivel de privacidad mínimo para cada tipo de sala de consulta en julio de 2024?

```SQL
WITH top_3_farmacias AS (
     SELECT pharmacy_name
     FROM fct_consultations
     WHERE ((EXTRACT(MONTH FROM consultation_date) = 7) AND
            (EXTRACT(YEAR FROM consultation_date) = 2024))
     GROUP BY pharmacy_name
     ORDER BY COUNT(consultation_id) ASC
     LIMIT 3
)
SELECT
    consultation_room_type,
    MIN(privacy_level_score) AS min_privacidad
FROM fct_consultations
WHERE pharmacy_name IN (SELECT top_3_farmacias.pharmacy_name FROM top_3_farmacias) AND
    ((EXTRACT(MONTH FROM consultation_date) = 7) AND
            (EXTRACT(YEAR FROM consultation_date) = 2024))
GROUP BY consultation_room_type;
```

In [39]:
res = df_consultation[
    (df_consultation['consultation_date'].dt.month == 7 ) &
    (df_consultation['consultation_date'].dt.year == 2024)
].groupby('pharmacy_name').agg(
    num_consultas = ('consultation_id', 'count')
).reset_index()

res = res.sort_values('num_consultas', ascending = True).head(3)

top_3_farmacias = res['pharmacy_name'].tolist()

res2 = df_consultation[
    (df_consultation['consultation_date'].dt.month == 7 ) &
    (df_consultation['consultation_date'].dt.year == 2024) &
    (df_consultation['pharmacy_name'].isin(top_3_farmacias))
].groupby('consultation_room_type').agg(
    min_privacidad = ('privacy_level_score', 'min')
).reset_index()

In [44]:
res = pl_consultation.filter(
    (pl.col('consultation_date').dt.month() == 7) &
    (pl.col('consultation_date').dt.year() == 2024)
).group_by('pharmacy_name').agg(
    pl.len().alias('num_consultas')
).sort('num_consultas', descending=False).head(3)

top_3_farmacias = res['pharmacy_name'].to_list()

res2 = pl_consultation.filter(
    (pl.col('consultation_date').dt.month() == 7) &
    (pl.col('consultation_date').dt.year() == 2024) &
    (pl.col('pharmacy_name').is_in(top_3_farmacias))
).group_by('consultation_room_type').agg(
    pl.col('privacy_level_score').min().alias('min_privacidad')
)

res2

consultation_room_type,min_privacidad
str,i64
"""private""",8
"""semi-private""",6
"""group""",4
"""open""",5
